# Biblical RAG LLM MVP 1

This notebook tests mistral:instruct's ability to read a Bible that is recursively chunked.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
import os
from uuid import uuid4

# Open King James Bible.
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
kjv_df = pd.read_csv(os.path.join(data_dir, 'en_kjv.csv'), index_col='index')
kjv_df.head()

,language,translation,book,chapter,verse,text
index,,,,,,
0,en,kjv,Gen,1,1,In the beginning God created the heaven and th...
1,en,kjv,Gen,1,2,"And the earth was without form, and void; and ..."
2,en,kjv,Gen,1,3,"And God said, Let there be light: and there wa..."
3,en,kjv,Gen,1,4,"And God saw the light, that it was good: and G..."
4,en,kjv,Gen,1,5,"And God called the light Day, and the darkness..."


In [2]:
# Open metadata table.
book_metadata = pd.read_csv(os.path.join(data_dir, 'metadata', 'books.csv'))
book_metadata.loc[book_metadata['book'].isin(['Numbers', 'Jonah', 'Ruth', 'Mark', 'Titus', 'Revelation'])]

,book,chapters,verses,avg verse per chapter,testament,category,author,abbreviation
3,Numbers,36,1288,36,old,law,Moses,Num
7,Ruth,4,85,21,old,writing,Samuel,Ruth
31,Jonah,4,48,12,old,prophet,Jonah,Jonah
40,Mark,16,678,42,new,gospel,Mark,Mark
55,Titus,3,46,15,new,epistle,Paul,Titus
65,Revelation,22,404,18,new,revelation,John,Rev


In [6]:
psalm_metadata = pd.read_csv(os.path.join(data_dir, 'metadata', 'psalms.csv'))
psalm_metadata.loc[psalm_metadata['psalm'].isin([1, 5, 42, 50, 89, 90, 127])]

,psalm,author
0,1,Anonymous
4,5,David
41,42,Sons of Korah
49,50,Asaph
88,89,Ethan
89,90,Moses
126,127,Solomon


## Prepare the Data

In [3]:
# Combine all verses into book chapters.
kjv_books = {}
book_title_list = book_metadata['abbreviation'].values
tween = ' ' # What goes between each verse
verse_total = book_metadata['verses'].sum() # Index of the final verse
print("Begin forming books...")
for book in book_title_list:
	full_title = book_metadata.loc[book_metadata['abbreviation'] == book]['book'].values[0]
	number_of_chapters = book_metadata.loc[book_metadata['abbreviation'] == book]['chapters'].values[0]
	for chapter in range(number_of_chapters):
		chapter_df = kjv_df.loc[(kjv_df['book'] == book) & (kjv_df['chapter'] == chapter+1)]
		full_title = book_metadata.loc[book_metadata['abbreviation'] == book]['book'].values[0]
		full_chapter_text = ''
		for index, row in chapter_df.iterrows():
			if row['chapter'] == chapter+1:
		   		full_chapter_text += row['text'] + tween
		kjv_books[full_title + ' | ' + str(chapter+1)] = full_chapter_text.rstrip() # Separate with a pipe for easy splitting later
print("Finished forming books!")
# Store in file
out_path = os.path.join(data_dir, 'en_kjv.json')
if not os.path.isfile(out_path):
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(kjv_books, f, indent=4, ensure_ascii=False)

Begin forming books...
Finished forming books!


In [4]:
# This method provides the chapter and verse citation for metadata.
def get_citation(text, df):
	citation = {} # Key is chapter number; value is list of verse numbers
	for index, row in df.iterrows():
		if row['text'] in text.page_content:
			if row['chapter'] in citation.keys():
				citation[row['chapter']].append(row['verse'])
			else:
				citation[row['chapter']] = [row['verse']]
	mykeys = []
	for key in citation.keys():
		mykeys.append(key)
	if len(mykeys) == 1:
		result = str(mykeys[0]) + ':' + str(citation[mykeys[0]][0]) + '-' + str(citation[mykeys[0]][-1])
	elif len(mykeys) >= 1:
		result = str(mykeys[0]) + ':' + str(citation[mykeys[0]][0]) + '-' + str(mykeys[-1]) + ':' + str(citation[mykeys[-1]][-1])
	else:
		result = ''
	return result

In [7]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough

local_llm = 'llama3'
dimensions=300

# Convert each book into a document.
def document_verses(doc_dict):
    documents = []
    for doc_index, doc in enumerate(doc_dict):
        title = doc.split(' | ')[0] # Some titles have a space (e.g. 1 Samuel), so the pipe is needed!
        if title == 'Psalms':
            author = psalm_metadata.loc[psalm_metadata['psalm'] == int(doc.split(' | ')[1])]['author'].values[0]
        else:
            author = book_metadata.loc[book_metadata['book'] == title]['author'].values[0]
        doc = Document(
            page_content=doc_dict[doc],
            metadata={
                'title': title,
                'citation': doc.replace(' | ', ' '),
                'author': author,
                'book_index': doc_index
            },
            id=str(uuid4())
        )
        documents.append(doc)
    return documents

# Split the documents.
split_docs = document_verses(kjv_books)
print("Documents split. There are {}.".format(len(split_docs)))

Documents split. There are 1189.


In [8]:
# Check the documents that have been split
'''print("There are {} chunks.\n".format(len(split_docs)))
for i in range(8):
    r = split_docs[i*100]
    print("{} ({}; by {})".format(r.page_content, r.metadata['citation'], r.metadata['author']))###### TO DO: CHAPTER AND VERRSE REFS ####
    print('')'''

'print("There are {} chunks.\n".format(len(split_docs)))\nfor i in range(8):\n    r = split_docs[i*100]\n    print("{} ({}; by {})".format(r.page_content, r.metadata[\'citation\'], r.metadata[\'author\']))###### TO DO: CHAPTER AND VERRSE REFS ####\n    print(\'\')'

## Store the Data

In [9]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_community.vectorstores import FAISS

print("Begin instantiating LLM.")
llm = OllamaLLM(model='mistral:instruct')
print("LLM instantiated.")
embedding = OllamaEmbeddings(model='mistral:instruct')
vectorstore = FAISS.from_documents(split_docs, embedding)
print("Vector store instantiated.")

Begin instantiating LLM.
LLM instantiated.
Vector store instantiated.


### Save & Load

In [ ]:
vector_file = os.path.join(data_dir, 'bible_vectostore_chapters')
vectorstore.save_local(vector_file)
print("File saved.")

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embedding = OllamaEmbeddings(model='mistral:instruct')
vector_file = os.path.join(data_dir, 'bible_vectostore_chapters')
vectorstore = FAISS.load_local(vector_file, embedding, allow_dangerous_deserialization=True)#.as_retriever() if you want to load it as a retriever
print("File loaded.")

# RAG Chain Tests
I will test several RAG chain settings below. There are 1189 Bible chapters in the RAG. First I will test gathering 1/10th & finally 1/100th of the chapters with MMR at 0.7, 0.5, and 0.3. Then I will choose the best one and try gathering fewer and fewer documents.

## Fetch 1/10th of the docs & give 1/100th of the docs with MMR search & lambda 0.7

In [10]:
from langchain.chains import RetrievalQA

print("Begin instantiating retriever.")
retriever = vectorstore.as_retriever(
    fetch_k=len(split_docs)//10,
    k=len(split_docs)//100,
    search_type='mmr',
    lambda_mult=0.7
)
print("Retriever instantiated.")
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)
print("RAG chain instantiated.")

Begin instantiating retriever.
Retriever instantiated.
RAG chain instantiated.


## Test the RAG Model

In [11]:
questions = [
    "Where did Jesus talk about wine?",
    "Who was Abraham's son?",
    "Where did God send Jonah? And where did Jonah try to go instead?",
    "Who are Luke's books addressed to?",
    "Which psalm says that angels will protect you even from dashing your foot against a stone?",
    "Why did God regret making Saul the King of Israel?",
    "What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?",
    "Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?",
    "When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?",
    "What is the genealogy of Enoch going back to Adam?",
    "Why did God flood the world in the time of Noah?",
    "What are the similarities between Daniel's prophesies and Revelation's prophesies?"
]
print("There are {} questions in the test suite.".format(len(questions)))

for question in questions:
    answer = qa.invoke(question)
    print(answer['query']+'\n'+answer['result']+'\n')

There are 12 questions in the test suite.
Where did Jesus talk about wine?
 In the New Testament, there are several instances where Jesus refers to or uses wine in His teachings. Here are a few examples:

1. At the Wedding at Cana (John 2:1-11) - This is one of the earliest accounts of Jesus's ministry and His first miracle. At a wedding, the hosts ran out of wine. Jesus told His disciples to fill six large jars with water, which they did. He then turned the water into wine, making it the best wine of the occasion.

2. The Parable of the Lost Sheep (Matthew 18:12-14) - In this parable, Jesus uses wine as a metaphor to illustrate God's joy in finding a lost soul and bringing them back to the fold. He says that there is more rejoicing in heaven over one sinner who repents than over ninety-nine righteous persons who do not need to repent.

3. The Last Supper (Matthew 26:26-29, Mark 14:22-25, Luke 22:17-20) - During the Last Supper with His disciples, Jesus instituted the Lord's Supper or 

## Assessment: RAG -> 10th -> 100th w/ MMR @ 0.7 lambda
### Where did Jesus talk about wine?
In 2 of the examples, Jesus doesn't talk about wine (the parable of the lost sheep, and the parable of the vinyard workers). It's missing times when Jesus said that he was called a drunkard and when he said that you don't put new wine in an old wineskin because the wineskin will burst.
### Who was Abraham's son?
The model gets the answer right, but mentions that the RAG docs did not mention Abraham and his sons.
### Where did God send Jonah? And where did Jonah try to go instead?
The model gets the answer right, but mentions that the RAG docs did not mention Jonah's journeys.
### Who are Luke's books addressed to?
The model fails to mention Theophilus. It says that the books were written for many people, which is a common theory.
### Which psalm says that angels will protect you even from dashing your foot against a stone?
The model correctly identifies Psalm 91, verse 12, but misquites the verse and says that the verse does not explicitly mention angels and tripping on stones. **This is a strange hallucination.**
### Why did God regret making Saul the King of Israel?
The model correctly identifies that Saul was rejected for disobedience, but does not mention the specific act of disobedience (choosing not to kill the enemy king and cattle). The RAG text did not mention Saul.
### What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?
The model correctly identifies circumcision of gentiles and following Jewish law as the issue of disagreement. It does not want to say who is "right", but acknowledges that the early church agreed with Paul.
### Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?
The model correctly identifies the vision Peter had, but says that the RAG model did not help.
### When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?
The model correctly identifies the reference to Numbers 21 and the bronze serpent statue.
### What is the genealogy of Enoch going back to Adam?
The model extracted the genealogy going to Noah (grandson of Enoch), which is more information than required. **This feels the most like a RAG extraction.**
### Why did God flood the world in the time of Noah?
The model correctly identifies the purpose of the flood.
### What are the similarities between Daniel's prophesies and Revelation's prophesies?
The model correctly identifies similarities between the 2 books.
### Score:
7/12
### RAG Fails:
4/12

## Fetch 1/10th of the docs & give 1/100th of the docs with MMR search & lambda 0.5

In [12]:
from langchain.chains import RetrievalQA

print("Begin instantiating retriever.")
retriever = vectorstore.as_retriever(
    fetch_k=len(split_docs)//10,
    k=len(split_docs)//100,
    search_type='mmr',
    lambda_mult=0.5
)
print("Retriever instantiated.")
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)
print("RAG chain instantiated.")

Begin instantiating retriever.
Retriever instantiated.
RAG chain instantiated.


## Test the RAG Model

In [13]:
questions = [
    "Where did Jesus talk about wine?",
    "Who was Abraham's son?",
    "Where did God send Jonah? And where did Jonah try to go instead?",
    "Who are Luke's books addressed to?",
    "Which psalm says that angels will protect you even from dashing your foot against a stone?",
    "Why did God regret making Saul the King of Israel?",
    "What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?",
    "Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?",
    "When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?",
    "What is the genealogy of Enoch going back to Adam?",
    "Why did God flood the world in the time of Noah?",
    "What are the similarities between Daniel's prophesies and Revelation's prophesies?"
]
print("There are {} questions in the test suite.".format(len(questions)))

for question in questions:
    answer = qa.invoke(question)
    print(answer['query']+'\n'+answer['result']+'\n')

There are 12 questions in the test suite.
Where did Jesus talk about wine?
 The New Testament of the Bible contains several references to Jesus talking about or using wine. One notable instance is at the Last Supper, where Jesus shared wine with His disciples (Matthew 26:27-29, Mark 14:23-25, Luke 22:17-20). Another example is when Jesus turned water into wine during a wedding in Cana (John 2:1-11). Additionally, there are other instances where Jesus mentions or uses the symbolism of wine to represent different aspects of His ministry and message (Matthew 9:17, Luke 5:37-39, Matthew 26:29, Mark 14:25, John 15:5). However, it's important to note that Jesus did not talk about wine in the way people often discuss it today, as a recreational or intoxicating beverage. Instead, He used it symbolically to illustrate spiritual concepts and truths.

Who was Abraham's son?
 Abraham did not have any sons mentioned in the given passage. The story revolves around Lot, who was a nephew to Abraham by

## Assessment RAG -> 10th -> 100th w/ MMR @ 0.5 lambda
### Where did Jesus talk about wine?
The model correctly identifies verses where Jesus talked about wine.
### Who was Abraham's son?
The model failed to answer the question, because the RAG model provided scriptures about Lot (Abraham's nephew).
### Where did God send Jonah? And where did Jonah try to go instead?
THe model correctly identidies Nineveh and Tarshish, but first says that the RAG texts are about Abraham and Sarah (more suitable to the previous question).
### Who are Luke's books addressed to?
The model fails to mention Theophilus. It says that the books were written for many people, which is a common theory.
### Which psalm says that angels will protect you even from dashing your foot against a stone?
The model correctly identifies Psalm 91, verse 12, and correctly quotes it.
### Why did God regret making Saul the King of Israel?
The model correctly identifies the reason for Saul's rejection, but the RAG texts are again about Abraham and Sarah. **Could the retrieval object be biased toward earlier texts?**
### What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?
The model correctly identifies the point of disagreement and where it is mentioned. The model does not want to say who is right, but mentions 2 common interpretations.
### Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?
The model correctly identified the vision Peter had, but says the RAG texts did not mention it. The model is also in denial about Peter's feeling of liberty to eat pork and shellfish.
### When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?
The model correctly identified Numbers 21 and interpreted it as a symbol for Jesus' crucifixion.
### What is the genealogy of Enoch going back to Adam?
The model gets the answer right, but is redundant and formats the answer in a bad way.
### Why did God flood the world in the time of Noah?
The model correctly identidies the purpose of the flood.
### What are the similarities between Daniel's prophesies and Revelation's prophesies?
The model found clear similarities between the 2 books.
### Score:
10/12
### RAG Fails:
4/12

## Fetch 1/10th of the docs & give 1/100th of the docs with MMR search & lambda 0.3

In [14]:
from langchain.chains import RetrievalQA

print("Begin instantiating retriever.")
retriever = vectorstore.as_retriever(
    fetch_k=len(split_docs)//10,
    k=len(split_docs)//100,
    search_type='mmr',
    lambda_mult=0.3
)
print("Retriever instantiated.")
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)
print("RAG chain instantiated.")

Begin instantiating retriever.
Retriever instantiated.
RAG chain instantiated.


## Test the RAG Model

In [15]:
questions = [
    "Where did Jesus talk about wine?",
    "Who was Abraham's son?",
    "Where did God send Jonah? And where did Jonah try to go instead?",
    "Who are Luke's books addressed to?",
    "Which psalm says that angels will protect you even from dashing your foot against a stone?",
    "Why did God regret making Saul the King of Israel?",
    "What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?",
    "Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?",
    "When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?",
    "What is the genealogy of Enoch going back to Adam?",
    "Why did God flood the world in the time of Noah?",
    "What are the similarities between Daniel's prophesies and Revelation's prophesies?"
]
print("There are {} questions in the test suite.".format(len(questions)))

for question in questions:
    answer = qa.invoke(question)
    print(answer['query']+'\n'+answer['result']+'\n')

There are 12 questions in the test suite.
Where did Jesus talk about wine?
 The Bible mentions wine in numerous places, but it's important to note that these passages do not specifically refer to Jesus himself talking about wine. Here are a few examples where wine is mentioned in the New Testament:

1. At the wedding at Cana (John 2:1-11) - This is one of the first miracles performed by Jesus, in which he turns water into wine to save the hosts from embarrassment.

2. The Last Supper (Matthew 26:26-29, Mark 14:22-25, Luke 22:17-20, and 1 Corinthians 11:23-26) - Jesus shared a cup of wine with his disciples during the Last Supper, symbolizing his blood that would be shed for humanity.

3. The parable of the wicked husbandmen (Matthew 21:33-46, Mark 12:1-12, and Luke 20:9-19) - In this parable, Jesus uses wine as a symbol to explain his relationship with Israel.

4. The parable of the great supper (Luke 14:15-24) - In this parable, a man invites many guests to a wedding feast but they al

## Assessment RAG -> 10th -> 100th w/ MMR @ 0.3 lambda
### Where did Jesus talk about wine?
The model fails to mention every instance where Jesus mentioned wine, and provides some where wine is not mentioned. The RAG model failed to help.
### Who was Abraham's son?
The model fails to mention Abraham's sons, and again was given the wrong RAG texts (about Lot).
### Where did God send Jonah? And where did Jonah try to go instead?
The model correctly identifies Nineveh and Tarshish, but the RAG texts are not about Jonah.
### Who are Luke's books addressed to?
The model correctly identifies Theophilus, but also says that the RAG texts provided were of the Old Testament.
### Which psalm says that angels will protect you even from dashing your foot against a stone?
The model correctly identifies Psalm 91, verse 12, but gets the quote wrong. **This is an odd hallucination.**
### Why did God regret making Saul the King of Israel?
The model correctly identifies the general reason for Saul's rejection. But again, the RAG text is about Sodom and Gomorah. **Why does the RAG model so often choose early Genesis chapters?!**
### What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?
The model correctly explains the issue, but is reluctant to say who was "right".
### Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?
The model correctly explains the issue, but is reluctant to take a firm stance.
### When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?
The model correctly cites and explains the reference of the bronze serpent. It does not interpret the reference.
### What is the genealogy of Enoch going back to Adam?
The model give a genealogy going all the way to Noah, Enoch's grandson.
### Why did God flood the world in the time of Noah?
The model correctly identifies the purpose of the flood.
### What are the similarities between Daniel's prophesies and Revelation's prophesies?
The model correctly identifies some similarities between the 2 books.
### Score:
9/12
### RAG Fails:
5/12

## Fetch 50 of the docs & give 5 of the docs with MMR search & lambda 0.8.

In [16]:
from langchain.chains import RetrievalQA

print("Begin instantiating retriever.")
retriever = vectorstore.as_retriever(
    fetch_k=len(split_docs)//10,
    k=len(split_docs)//100,
    search_type='mmr',
    lambda_mult=0.8
)
print("Retriever instantiated.")
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)
print("RAG chain instantiated.")

Begin instantiating retriever.
Retriever instantiated.
RAG chain instantiated.


## Test the RAG Model

In [17]:
questions = [
    "Where did Jesus talk about wine?",
    "Who was Abraham's son?",
    "Where did God send Jonah? And where did Jonah try to go instead?",
    "Who are Luke's books addressed to?",
    "Which psalm says that angels will protect you even from dashing your foot against a stone?",
    "Why did God regret making Saul the King of Israel?",
    "What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?",
    "Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?",
    "When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?",
    "What is the genealogy of Enoch going back to Adam?",
    "Why did God flood the world in the time of Noah?",
    "What are the similarities between Daniel's prophesies and Revelation's prophesies?"
]
print("There are {} questions in the test suite.".format(len(questions)))

for question in questions:
    answer = qa.invoke(question)
    print(answer['query']+'\n'+answer['result']+'\n')

There are 12 questions in the test suite.
Where did Jesus talk about wine?
 The question asks about Jesus talking about wine, but the provided text does not include any mention of Jesus or the New Testament. In the New Testament, there are numerous instances where Jesus talks about, mentions, or uses wine in his teachings and parables. For example, in John 2:1-11, Jesus performs his first miracle by turning water into wine at a wedding in Cana of Galilee. In Matthew 26:26-29, Jesus institutes the Eucharist by saying, "This is my body, which is given for you. Do this in remembrance of me." In the same passage, he also says, "This cup that is poured out for you is the new covenant in my blood."

Who was Abraham's son?
 The question does not explicitly mention a specific son of Abraham, but it is important to note that Isaac was the only direct son mentioned in the passage provided and he was later considered as Abraham's primary son. However, this text focuses on the story of Lot, so the

## Assessment RAG -> 50 -> 5 w/ MMR @ 0.5 lambda
### Where did Jesus talk about wine?
The model mentions only 2 places where Jesus talked about wine. It also says the RAG model failed to provide New Testament texts.
### Who was Abraham's son?
The model mentions Isaac, but says that the RAG texts given were about Lot.
### Where did God send Jonah? And where did Jonah try to go instead?
The model mentions Nineveh and Tarshish, but says that the RAG texts given were about Abraham and Sarah.
### Who are Luke's books addressed to?
The model fails to mention Theophilus, and says the RAG model provided Old Testament texts.
### Which psalm says that angels will protect you even from dashing your foot against a stone?
The model correctly identifies Psalm 91, verse 12, but falsely quotes verse 14. **Is this a RAG hallucination?**
### Why did God regret making Saul the King of Israel?
The model mentions the portion of 1 Samuel that explains Saul's disobedience, but says that the RAG texts were from Genesis.
### What did Paul disagree with Peter about according to Paul's letter to the Galatians? Who was right?
The model correctly explains the issue of circumcision, but does not want to say who was "right".
### Why did Peter feel comfortable eating pork and shellfish after receiving the Holy Spirit?
The model correctly explains why Peter felt comfortable, but mentions the RAG texts are from the Old Testament.
### When Jesus mentions that Moses lifted the serpent in the wilderness (John 3:14), what is He referencing?
The model correctly identifies Numbers 21, and even mentions that Jesus was talking to Nicodemus in John 3:14.
### What is the genealogy of Enoch going back to Adam?
The model correctly lists the genealogy going all the way to Noah, Enoch's grandson.
### Why did God flood the world in the time of Noah?
The model clearly extracted the portion of Genesis about the flood. **The RAG selection is biased toward this area, but we know extraction works.**
### What are the similarities between Daniel's prophesies and Revelation's prophesies?
This is the best explanation of the similarities out of the 4 conditions.
### Score:
9/12
### RAG Fails:
6/12